In [4]:
def confusion_matrix(y_true, y_pred, labels):
    """
    Build a confusion matrix as a nested dictionary.
    
    Args:
        y_true: List of true labels
        y_pred: List of predicted labels
        labels: List of all possible label values
    
    Returns:
        Nested dictionary: {true_label: {pred_label: count}}
    """
    # Initialize the nested dictionary with zeros
    cm = {true_label: {pred_label: 0 for pred_label in labels} for true_label in labels}
    
    # Count predictions
    for true, pred in zip(y_true, y_pred):
        if true in cm and pred in cm[true]:
            cm[true][pred] += 1
        else:
            # Handle unexpected labels gracefully
            print(f"Warning: Unexpected label pair ({true}, {pred})")
    
    return cm

def pretty_print_cm(cm, labels):
    """
    Print confusion matrix as a formatted grid.
    
    Args:
        cm: Confusion matrix dictionary
        labels: List of labels in order
    """
    print("\n" + "=" * 60)
    print("CONFUSION MATRIX")
    print("=" * 60)
    
    # Print header
    print(f"{'True\\Pred':<12}", end="")
    for label in labels:
        print(f"{label:>8}", end="")
    print("\n" + "-" * 60)
    
    # Print each row
    for true_label in labels:
        print(f"{true_label:<12}", end="")
        for pred_label in labels:
            count = cm.get(true_label, {}).get(pred_label, 0)
            print(f"{count:>8}", end="")
        print()
    
    print("=" * 60)

def calculate_per_class_metrics(cm, labels):
    """
    Calculate per-class precision and recall from confusion matrix.
    
    Returns:
        Dictionaries: {'label': {'precision': value, 'recall': value}}
    """
    metrics = {}
    
    for label in labels:
        # Get values for this class
        TP = cm.get(label, {}).get(label, 0)
        
        # Precision: TP / (TP + FP)
        # FP = sum of predictions for this label from other true classes
        FP = 0
        for other_label in labels:
            if other_label != label:
                FP += cm.get(other_label, {}).get(label, 0)
        
        # Recall: TP / (TP + FN)
        # FN = sum of predictions for other labels when true is this label
        FN = 0
        for other_label in labels:
            if other_label != label:
                FN += cm.get(label, {}).get(other_label, 0)
        
        # Calculate metrics
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        
        metrics[label] = {
            'precision': precision,
            'recall': recall,
            'TP': TP,
            'FP': FP,
            'FN': FN
        }
    
    return metrics

# Generate test data (15+ samples)
import random

# Set seed for reproducibility
random.seed(42)

labels = ['cat', 'dog', 'bird']

# Create realistic predictions with some errors
y_true = []
y_pred = []

# Generate 20 samples with some intentional mistakes
sample_data = [
    ('cat', 'cat'), ('cat', 'cat'), ('cat', 'dog'), ('cat', 'cat'), ('cat', 'bird'),
    ('dog', 'dog'), ('dog', 'dog'), ('dog', 'cat'), ('dog', 'dog'), ('dog', 'dog'),
    ('bird', 'bird'), ('bird', 'bird'), ('bird', 'cat'), ('bird', 'bird'), ('bird', 'bird'),
    ('cat', 'cat'), ('dog', 'dog'), ('bird', 'bird'), ('cat', 'cat'), ('dog', 'dog')
]

for true, pred in sample_data:
    y_true.append(true)
    y_pred.append(pred)

print("📊 GENERATING CONFUSION MATRIX")
print(f"Number of samples: {len(y_true)}")
print(f"Labels: {labels}")
print(f"\nFirst 10 predictions:")
for i in range(10):
    print(f"  Sample {i+1}: True={y_true[i]}, Pred={y_pred[i]}")

# Build confusion matrix
cm = confusion_matrix(y_true, y_pred, labels)

# Print pretty confusion matrix
pretty_print_cm(cm, labels)

# Calculate and display per-class metrics
metrics = calculate_per_class_metrics(cm, labels)

print("\n" + "=" * 60)
print("PER-CLASS METRICS")
print("=" * 60)
print(f"{'Class':<10} {'Precision':<12} {'Recall':<12} {'TP':<6} {'FP':<6} {'FN':<6}")
print("-" * 60)

for label in labels:
    prec = metrics[label]['precision']
    rec = metrics[label]['recall']
    tp = metrics[label]['TP']
    fp = metrics[label]['FP']
    fn = metrics[label]['FN']
    print(f"{label:<10} {prec:<12.4f} {rec:<12.4f} {tp:<6} {fp:<6} {fn:<6}")

# Calculate overall accuracy
total_correct = sum(cm[label][label] for label in labels)
total_samples = len(y_true)
accuracy = total_correct / total_samples

print("-" * 60)
print(f"\n🎯 Overall Accuracy: {accuracy:.4f} ({total_correct}/{total_samples})")

# Analyze model performance
print("\n" + "=" * 60)
print("MODEL PERFORMANCE ANALYSIS")
print("=" * 60)

for label in labels:
    prec = metrics[label]['precision']
    rec = metrics[label]['recall']
    
    if prec < 0.7:
        print(f"⚠️  {label.upper()}: Low precision ({prec:.2f}) - many false positives")
    elif rec < 0.7:
        print(f"⚠️  {label.upper()}: Low recall ({rec:.2f}) - many false negatives")
    else:
        print(f"✅ {label.upper()}: Good performance (P={prec:.2f}")

📊 GENERATING CONFUSION MATRIX
Number of samples: 20
Labels: ['cat', 'dog', 'bird']

First 10 predictions:
  Sample 1: True=cat, Pred=cat
  Sample 2: True=cat, Pred=cat
  Sample 3: True=cat, Pred=dog
  Sample 4: True=cat, Pred=cat
  Sample 5: True=cat, Pred=bird
  Sample 6: True=dog, Pred=dog
  Sample 7: True=dog, Pred=dog
  Sample 8: True=dog, Pred=cat
  Sample 9: True=dog, Pred=dog
  Sample 10: True=dog, Pred=dog

CONFUSION MATRIX
True\Pred        cat     dog    bird
------------------------------------------------------------
cat                5       1       1
dog                1       6       0
bird               1       0       5

PER-CLASS METRICS
Class      Precision    Recall       TP     FP     FN    
------------------------------------------------------------
cat        0.7143       0.7143       5      2      2     
dog        0.8571       0.8571       6      1      1     
bird       0.8333       0.8333       5      1      1     
-------------------------------------------